# Train / validation / test: re-checking the results and trying new letter models

Three things are done here.

1. **A proper split.** The data is divided into training, validation and test rows (`splits.py`,
   written out to `data_splits.csv`). Models and settings are chosen on validation rows only; test
   rows are scored once.
2. **The earlier numbers are recomputed** and compared with what the README and `EXPERIMENTS.md` say.
3. **New models are compared with ExtraTrees** on letter recognition: more tree models, boosting,
   a PyTorch neural network, a graph network on the hand skeleton, a wider feature set, label
   cleaning, extra training data, and blends.

There are two different questions, and they get different splits and very different answers:

| | question | rows tested | how the split is made |
|---|---|---|---|
| **A. New person** | how well is a signer recognised whom the model has never seen? | the 5 dataset signers, 40 letters | whole signers: 3 train, 1 validation, 1 test, rotated so each signer is the test signer once |
| **B. Calibrated user** | how well is a person recognised after recording a few takes of each letter? | the calibration takes of Hassan and Abdullah, 30 letters | whole takes, per letter: 60% train, 20% validation, 20% test; the dataset signers are all training |

A random split of all rows would mix these and put the same person on both sides.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, os, time
import numpy as np, pandas as pd
from scipy import stats
import evaluation as ev, splits, models, letter_models as lm

pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)
P, M = ev.load_dataset(include_users=True)
y, sg = M.letter.to_numpy(), M.signer.to_numpy()
user = splits.is_user(sg)
OUT = "results/2026-11-15_train_val_test"; os.makedirs(OUT, exist_ok=True)
print(f"{len(M)} rows: {(~user).sum()} dataset holds from {len(set(sg[~user]))} signers, "
      f"{user.sum()} calibration takes; {len(set(y))} letters")

1636 rows: 1345 dataset holds from 5 signers, 291 calibration takes; 41 letters


## 1. The split

One row is one hold (dataset) or one take (calibration; its middle frame).

In [2]:
T = splits.split_table(M, seed=0)
T.to_csv("data_splits.csv", index=False)
ROT = splits.signer_rotations(sg)
print("A. New person: signers per rotation")
print(pd.DataFrame([{"rotation": k, "train": ", ".join(tr), "validation": va, "test": te,
                     "rows train / val / test": " / ".join(str((T[f'new_person_rotation{k}'] == r).sum()) for r in ("train", "val", "test"))}
                    for k, (tr, va, te) in enumerate(ROT, 1)]).to_string(index=False))
print("\nB. Calibrated user: rows per role")
cu = T.calibrated_user
print(pd.DataFrame({"dataset holds": cu[~user].value_counts(), "calibration takes": cu[user].value_counts()})
        .reindex(["train", "val", "test"]).fillna(0).astype(int).to_string())
letters_cal = sorted(set(y[user]))
print(f"\ncalibration letters ({len(letters_cal)}): {' '.join(letters_cal)}")
print("letters with no calibration takes:", " ".join(sorted(set(y) - set(letters_cal))))

A. New person: signers per rotation
 rotation                           train validation      test rows train / val / test
        1 signer3-F, signer4-L, signer5-R  signer2-F signer1-F         732 / 302 / 311
        2 signer1-F, signer4-L, signer5-R  signer3-F signer2-F         764 / 279 / 302
        3 signer1-F, signer2-F, signer5-R  signer4-L signer3-F         771 / 295 / 279
        4 signer1-F, signer2-F, signer3-F  signer5-R signer4-L         892 / 158 / 295
        5 signer2-F, signer3-F, signer4-L  signer1-F signer5-R         876 / 311 / 158

B. Calibrated user: rows per role
                 dataset holds  calibration takes
calibrated_user                                  
train                     1345                175
val                          0                 58
test                         0                 58

calibration letters (30): ء ا ب ت ث ح خ د ذ ر ز س ص ض ط ظ ع غ ف ق ل و پ چ ڈ ژ ک گ ی ے
letters with no calibration takes: آ ؤ ئ ج ش م ن ٹ ڑ ھ ہ


## 2. Are the earlier results still right?

**Whole notebooks, re-run from scratch on 15 Nov 2026** (fresh copies, the stored outputs compared line by line):

| notebook | what it holds | outcome |
|---|---|---|
| `headline_statistics.ipynb` | SVM letter accuracy, intervals, nested tuning, shuffled-label null | every number identical, except the largest of the 30 shuffled-label runs: 5.1% against 4.9% stored (that control is unseeded) |
| `model_alternatives.ipynb` | six letter models, nested selection, end-to-end words per model | every number identical (only run times differ) |
| `alignment_decoder.ipynb` | word decoder ablation, seeds, controls | every number identical, except the shuffled-label control: 5.6% top-1 against 6.9% stored (unseeded; the README quotes 7.2% from a third run) |
| `end_to_end_words.ipynb` | word accuracy, unaided against known letter count | every number identical |

So the headline results reproduce exactly. The one loose end is the shuffled-label control, which
gives 5.6 to 7.2% depending on the run; it should be quoted as "about 6%", or given a seed.

The cell below recomputes the main letter numbers again, independently of those notebooks, with the
code used in the rest of this notebook.

In [3]:
D = ~user                                   # dataset rows
CLS = sorted(set(y))

def macro(ok, yy):                          # mean of per-letter recall
    return 100 * pd.Series(np.asarray(ok, float)).groupby(np.asarray(yy)).mean().mean()

def loso(make, extra=None, seed=0):
    # leave-one-signer-out over the dataset signers; `extra` rows are added to every training fold
    pred = np.empty(len(y), dtype=object)
    for s in sorted(set(sg[D])):
        tr = D & (sg != s)
        if extra is not None: tr = tr | extra
        pred[sg == s] = make(seed).fit(P[tr], y[tr], groups=sg[tr]).predict(P[sg == s])
    return pred[D]

yD, sD = y[D], sg[D]
rows = []
def check(what, stored, got, tol=0.06):
    rows.append({"result": what, "stored": stored, "recomputed": round(got, 1), "same": "yes" if abs(got - stored) < tol else "NO"})

for name, acc, mac in [("SVM-RBF (shipped)", 26.5, 23.7), ("ExtraTrees", 31.7, 26.7), ("MLP 128, alpha=3", 33.5, 17.4),
                       ("LDA, shrinkage", 30.6, 22.6), ("kNN, k=15", 31.1, 17.7), ("ensemble: SVM+MLP+ExtraTrees+LDA", 33.2, 23.0)]:
    a, m = [], []
    for seed in range(3):                   # the stored numbers are means over seeds 0-2
        ok = loso(lm.CANDIDATES[name][1], seed=seed) == yD
        a.append(100 * ev.per_fold(ok.astype(float), sD).mean()); m.append(macro(ok, yD))
    check(f"{name}: accuracy, leave-one-signer-out", acc, np.mean(a))
    check(f"{name}: macro recall", mac, np.mean(m))

svm = lm.CANDIDATES["SVM-RBF (shipped)"][1]
hassan = user & (M.signer.str.startswith("user-hassan").to_numpy())
for label, extra, stored in [("dataset signers only", None, 26.4), ("+ Hassan's 30 earlier takes in training", hassan, 26.8),
                             ("+ all 291 calibration takes in training", user, 28.0)]:
    check(f"SVM pooled accuracy on dataset signers, {label}", stored, 100 * (loso(svm, extra) == yD).mean())
print(pd.DataFrame(rows).to_string(index=False))

                                                                         result  stored  recomputed same
                              SVM-RBF (shipped): accuracy, leave-one-signer-out    26.5        26.5  yes
                                                SVM-RBF (shipped): macro recall    23.7        23.7  yes
                                     ExtraTrees: accuracy, leave-one-signer-out    31.7        31.7  yes
                                                       ExtraTrees: macro recall    26.7        26.7  yes
                               MLP 128, alpha=3: accuracy, leave-one-signer-out    33.5        33.5  yes
                                                 MLP 128, alpha=3: macro recall    17.4        17.4  yes
                                 LDA, shrinkage: accuracy, leave-one-signer-out    30.6        30.6  yes
                                                   LDA, shrinkage: macro recall    22.6        22.6  yes
                                      kNN, k=15: accura

**The 13 Nov calibration audit** (`results/2026-11-13_calibration_audit/`) reported 241 of 261 takes right when
each take is tested by a model that saw the other takes of the same sitting (five folds, whole takes held out,
every frame of the take voting). Recomputed here with the same folds:

In [4]:
FILES = {"user-hassan": "2026-10-22", "user-hassan-session2": "2026-10-26", "user-hassan-session3": "2026-11-11",
         "user-hassan-session4": "2026-11-12", "user-mixed-abdullah-hassan-2026-11-13": "2026-11-13"}
frames = {}
for signer, d in FILES.items():
    for t in json.load(open(f"calibration/calibration_{d}.json", encoding="utf-8"))["takes"]:
        frames[(signer, f"calib_{t['letter']}_{t['take']}")] = np.array([f["pts"] for f in t["frames"]], dtype=np.float32)
FR = [frames[(s, v)] if u else None for s, v, u in zip(M.signer, M.video, user)]   # every frame of each take

def take_predict(m, idx):
    # one prediction per take: the model's scores averaged over all its frames (the live page's rule)
    return np.array([m.classes_[m.predict_proba(FR[i]).mean(0).argmax()] for i in idx], dtype=object)

mixed = np.where(sg == "user-mixed-abdullah-hassan-2026-11-13")[0]
fold, seen = [], {}
for i in mixed:                              # round-robin by take order within each letter, as in the audit
    fold.append(seen.get(y[i], 0) % 5); seen[y[i]] = seen.get(y[i], 0) + 1
fold = np.array(fold); right = 0
prior = ~(sg == "user-mixed-abdullah-hassan-2026-11-13")
for k in range(5):
    tr = prior.copy(); tr[mixed[fold != k]] = True
    m = svm(0).fit(P[tr], y[tr])
    right += (take_predict(m, mixed[fold == k]) == y[mixed[fold == k]]).sum()
print(f"same-session, five folds, SVM: {right}/261 = {100*right/261:.1f}%   (stored: 241/261 = 92.3%)")

same-session, five folds, SVM: 241/261 = 92.3%   (stored: 241/261 = 92.3%)


## 3. Split A: a new person

For each rotation every model is fitted on the three training signers and scored on the validation
signer and the test signer. It is then refitted on training + validation (four signers) and scored
on the test signer again; that last number is the one comparable with the earlier
leave-one-signer-out results.

Two measures, because they disagree:

* **accuracy**: share of holds right, mean of the five signers. Dominated by the common letters.
* **macro recall**: mean of per-letter recall over the 40 letters. Rare letters count as much as common ones.

"earlier" models were already compared in `model_alternatives.ipynb`; "new" ones are tried here for the first time.

In [5]:
def new_person(make, extra=None, seed=0):
    # correctness of every dataset row as validation row, as test row (trained on 3) and as test row (trained on 4)
    val = np.zeros(len(y)); t3 = np.zeros(len(y)); t4 = np.zeros(len(y))
    for train, va_s, te_s in ROT:
        tr = np.isin(sg, train); va = sg == va_s; te = sg == te_s
        fit = lambda rows: make(seed).fit(P[rows | extra] if extra is not None else P[rows],
                                          y[rows | extra] if extra is not None else y[rows],
                                          groups=sg[rows | extra] if extra is not None else sg[rows])
        m = fit(tr)
        val[va] = m.predict(P[va]) == y[va]; t3[te] = m.predict(P[te]) == y[te]
        t4[te] = fit(tr | va).predict(P[te]) == y[te]
    return val[D], t3[D], t4[D]

acc = lambda ok: 100 * ev.per_fold(ok, sD).mean()
RES, rows = {}, []
def run(name, group, make, extra=None):
    t = time.time(); RES[name] = new_person(make, extra); v, t3, t4 = RES[name]
    rows.append({"model": name, "group": group, "val acc": acc(v), "val macro": macro(v, yD),
                 "test acc (3 signers)": acc(t3), "test macro (3)": macro(t3, yD),
                 "test acc (4 signers)": acc(t4), "test macro (4)": macro(t4, yD), "seconds": round(time.time() - t)})

for name, (group, make) in lm.CANDIDATES.items():
    run(name, group, make)
# the same models with the 291 calibration takes (other people, labels known) added to every training set
for name in ("ExtraTrees", "ExtraTrees, rich features", "Neural net (PyTorch MLP)"):
    run(name + " + calibration takes", "new, more data", lm.CANDIDATES[name][1], extra=user)
TA = pd.DataFrame(rows).set_index("model")
TA.to_csv(f"{OUT}/new_person.csv")
print(TA.round(1).to_string())

                                                        group  val acc  val macro  test acc (3 signers)  test macro (3)  test acc (4 signers)  test macro (4)  seconds
model                                                                                                                                                                 
SVM-RBF (shipped)                                     earlier     22.1       19.5                  23.3            19.8                  26.5            23.7       13
ExtraTrees                                            earlier     28.4       24.5                  28.0            21.3                  31.8            27.3       24
MLP 128, alpha=3                                      earlier     30.5       16.5                  30.8            17.6                  33.7            17.4       77
LDA, shrinkage                                        earlier     28.9       21.1                  28.9            20.8                  30.6            22.6        

### Choosing on validation, then testing

Picking the best row of the table above by its test column would be choosing on the test set. So the
choice is made per rotation: the model that scores best on that rotation's **validation signer** is
refitted on the four non-test signers and scored on the **test signer**. The validation signer plays no
part in the test.

In [6]:
names = list(RES)
ref = RES["ExtraTrees"][2]
def choose(criterion):
    ok = np.zeros(len(yD)); picked = {}
    for train, va_s, te_s in ROT:
        va, te = sD == va_s, sD == te_s
        score = {n: (RES[n][0][va].mean() if criterion == "accuracy" else macro(RES[n][0][va], yD[va])) for n in names}
        best = max(score, key=score.get); picked[te_s] = best
        ok[te] = RES[best][2][te]
    return ok, picked

sel_rows = []
for crit in ("accuracy", "macro recall"):
    ok, picked = choose(crit)
    r = ev.summarise(f"chosen by validation {crit}", ok, sD, baseline=ref)
    sel_rows.append({"how chosen": f"validation {crit}", "test accuracy %": r["mean of folds %"], "interval lo": r["boot lo %"],
                     "interval hi": r["boot hi %"], "test macro recall %": macro(ok, yD),
                     "vs ExtraTrees (pts)": r["vs baseline (pts)"], "paired p": r["paired t p"]})
    print(f"chosen by validation {crit}:")
    for s, n in picked.items(): print(f"    test signer {s}: {n}")
r = ev.summarise("ExtraTrees", ref, sD)
sel_rows.append({"how chosen": "ExtraTrees, no choice (reference)", "test accuracy %": r["mean of folds %"], "interval lo": r["boot lo %"],
                 "interval hi": r["boot hi %"], "test macro recall %": macro(ref, yD)})
print(); print(pd.DataFrame(sel_rows).round(3).to_string(index=False))

chosen by validation accuracy:
    test signer signer1-F: ExtraTrees, rich features + calibration takes
    test signer signer2-F: ExtraTrees, rich features + calibration takes
    test signer signer3-F: MLP 128, alpha=3
    test signer signer4-L: ensemble: SVM+MLP+ExtraTrees+LDA
    test signer signer5-R: 5-model blend (geometric mean)
chosen by validation macro recall:
    test signer signer1-F: ExtraTrees, rich features + calibration takes
    test signer signer2-F: ExtraTrees, rich features + calibration takes
    test signer signer3-F: ExtraTrees + calibration takes
    test signer signer4-L: ExtraTrees + calibration takes
    test signer signer5-R: 5-model blend (geometric mean)

                       how chosen  test accuracy %  interval lo  interval hi  test macro recall %  vs ExtraTrees (pts)  paired p
              validation accuracy           33.706       30.657       36.667               26.096                1.935     0.090
          validation macro recall           32.

### Each model against ExtraTrees, on the test signers

Difference in test accuracy (trained on four signers), paired over the five signers. With five signers,
a difference needs to be large and consistent to count; "wins" is the number of signers on which the model
is ahead of ExtraTrees.

In [7]:
cmp_rows = []
for n in names:
    ok = RES[n][2]; d = ev.per_fold(ok - ref, sD)
    lo, hi = ev.bootstrap_by_signer(ok, sD)
    cmp_rows.append({"model": n, "test acc %": acc(ok), "interval": f"{100*lo:.1f} to {100*hi:.1f}", "macro recall %": macro(ok, yD),
                     "acc vs ExtraTrees": 100 * d.mean(), "wins of 5": int((d > 0).sum()),
                     "paired p": stats.ttest_1samp(d, 0).pvalue if n != "ExtraTrees" else np.nan,
                     "macro vs ExtraTrees": macro(ok, yD) - macro(ref, yD)})
CMP = pd.DataFrame(cmp_rows).set_index("model").sort_values("test acc %", ascending=False)
print(CMP.round(3).to_string())

                                               test acc %      interval  macro recall %  acc vs ExtraTrees  wins of 5  paired p  macro vs ExtraTrees
model                                                                                                                                               
MLP 128, alpha=3                                   33.652  31.1 to 36.1          17.385              1.881          4     0.189               -9.932
5-model blend (geometric mean)                     33.522  30.3 to 36.9          27.166              1.751          4     0.070               -0.151
ensemble: SVM+MLP+ExtraTrees+LDA                   33.363  31.5 to 35.6          23.113              1.593          3     0.308               -4.204
ExtraTrees, cleaned labels                         33.064  29.8 to 36.3          23.674              1.293          4     0.179               -3.642
ExtraTrees, rich features + calibration takes      32.117  29.1 to 35.6          28.870              0.347

### Run-to-run spread

Tree and neural models depend on a random seed. The reference and the strongest new candidates, three seeds each:

In [8]:
top = ["ExtraTrees", "ExtraTrees, rich features", "ExtraTrees, rich features + calibration takes",
       "ExtraTrees + neural net", "5-model blend (geometric mean)"]
srows = []
for n in top:
    base = n.replace(" + calibration takes", "")
    extra = user if n.endswith("calibration takes") else None
    a, mc = [], []
    for seed in range(3):
        ok = RES[n][2] if seed == 0 else new_person(lm.CANDIDATES[base][1], extra, seed)[2]
        a.append(acc(ok)); mc.append(macro(ok, yD))
    srows.append({"model": n, "test acc, mean of 3 seeds": np.mean(a), "lowest": min(a), "highest": max(a),
                  "macro recall, mean": np.mean(mc), "lowest ": min(mc), "highest ": max(mc)})
SEEDS_A = pd.DataFrame(srows).set_index("model"); print(SEEDS_A.round(1).to_string())

                                               test acc, mean of 3 seeds  lowest  highest  macro recall, mean  lowest   highest 
model                                                                                                                           
ExtraTrees                                                          31.7    31.7     31.8                26.7     26.0      27.3
ExtraTrees, rich features                                           31.0    30.9     31.1                26.8     26.0      27.9
ExtraTrees, rich features + calibration takes                       32.3    32.1     32.5                29.4     28.9      30.1
ExtraTrees + neural net                                             28.9    28.5     29.2                21.6     20.9      22.3
5-model blend (geometric mean)                                      33.3    33.1     33.5                26.9     26.8      27.2


### Using the new person's unlabelled holds (transductive)

These standardise each signer's features by that signer's own average, including the test signer's. No test
labels are used, but the test signer's holds are. It is a different, easier setting, so it is not ranked with the others.

In [9]:
trows = []
for name, make in lm.TRANSDUCTIVE.items():
    v, t3, t4 = new_person(make)
    d = ev.per_fold(t4 - ref, sD)
    trows.append({"model": name, "val acc": acc(v), "test acc (4 signers)": acc(t4), "test macro (4)": macro(t4, yD),
                  "acc vs ExtraTrees": 100 * d.mean(), "wins of 5": int((d > 0).sum()), "paired p": stats.ttest_1samp(d, 0).pvalue})
print(pd.DataFrame(trows).set_index("model").round(3).to_string())

                                        val acc  test acc (4 signers)  test macro (4)  acc vs ExtraTrees  wins of 5  paired p
model                                                                                                                        
ExtraTrees rich, per-signer normalised   29.206                32.251          27.466              0.480          4     0.438
LDA, per-signer normalised               31.142                32.646          23.925              0.875          3     0.444


## 4. Split B: a calibrated user

Training: all 1,345 dataset holds plus the training takes of the calibration recordings. Validation and test:
calibration takes the model has not seen. Two ways of scoring a take:

* **one frame**: the take's middle frame only, the same kind of row the model is trained on.
* **whole take**: the model's scores averaged over every frame of the take, which is what the live page does.

The fixed split (seed 0) is the one in `data_splits.csv`. Because its test set is only 58 takes, the same
comparison is repeated on four more random splits and the five are averaged.

In [10]:
CAL = {"SVM-RBF (shipped), browser vote share": lambda s=0: lm.SvmVoteShare(s),
       "SVM-RBF, Platt probabilities": lambda s=0: lm.OnFeatures("geom", models.svm(s))}
CAL.update({n: mk for n, (g, mk) in lm.CANDIDATES.items()
            if n not in ("SVM-RBF (shipped)", "ExtraTrees, cleaned labels")})   # cleaning targets the dataset labels
SPLITS = range(5)
recs, PRED = [], {}
for name, make in CAL.items():
    t = time.time()
    for seed in SPLITS:
        role = splits.calibration_split(M, seed)
        tr = (~user) | (role == "train")
        m = make(0).fit(P[tr], y[tr], groups=sg[tr])
        for part in ("val", "test"):
            idx = np.where(role == part)[0]
            one = m.predict(P[idx]) == y[idx]; whole = take_predict(m, idx) == y[idx]
            PRED[(name, seed, part)] = (idx, one, whole)
            recs.append({"model": name, "split": seed, "part": part, "n": len(idx), "one frame %": 100 * one.mean(), "whole take %": 100 * whole.mean()})
    print(f"{name:40s} {time.time()-t:5.0f}s")
RB = pd.DataFrame(recs); RB.to_csv(f"{OUT}/calibrated_user.csv", index=False)

SVM-RBF (shipped), browser vote share       24s


SVM-RBF, Platt probabilities                24s


ExtraTrees                                  68s


MLP 128, alpha=3                            25s


LDA, shrinkage                               1s


kNN, k=15                                    5s


... (7 outputs left out)

Neural net (PyTorch MLP), rich             104s


Graph network on the hand skeleton         168s


ExtraTrees + neural net                    165s


5-model blend (geometric mean)             186s


In [11]:
def table(sel):
    return sel.pivot_table(index="model", columns="part", values=["one frame %", "whole take %"], sort=False)[
        [("one frame %", "val"), ("one frame %", "test"), ("whole take %", "val"), ("whole take %", "test")]]
print("Fixed split (seed 0): 58 validation takes, 58 test takes")
B0 = table(RB[RB.split == 0]); print(B0.round(1).to_string())
print("\nMean of five random splits")
B5 = table(RB.groupby(["model", "part"], sort=False)[["one frame %", "whole take %"]].mean().reset_index()); print(B5.round(1).to_string())

Fixed split (seed 0): 58 validation takes, 58 test takes
                                      one frame %       whole take %      
part                                          val  test          val  test
model                                                                     
SVM-RBF (shipped), browser vote share        82.8  87.9         93.1  86.2
SVM-RBF, Platt probabilities                 74.1  70.7         74.1  75.9
ExtraTrees                                   89.7  91.4         96.6  93.1
MLP 128, alpha=3                             81.0  79.3         82.8  81.0
LDA, shrinkage                               82.8  81.0         84.5  86.2
kNN, k=15                                    77.6  75.9         77.6  77.6
ensemble: SVM+MLP+ExtraTrees+LDA             89.7  87.9         91.4  89.7
Random forest                                93.1  89.7         94.8  84.5
Gradient boosting (HistGB)                   86.2  89.7         89.7  86.2
Logistic regression                        

### Choosing on validation, then testing

On each split the model with the best validation score is picked and its test score read off. Then every model
is compared with ExtraTrees on the same test takes, pooled over the five splits (a take can be a test take in
more than one split, so the pooled counts are not independent; the paired count of takes one model gets right and
the other wrong is the fairer comparison).

In [12]:
for measure, col in (("one frame", 1), ("whole take", 2)):
    picks, scores = [], []
    for seed in SPLITS:
        best = max(CAL, key=lambda n: (PRED[(n, seed, "val")][col].mean(), PRED[(n, seed, "val")][3 - col].mean()))
        picks.append(best); scores.append(100 * PRED[(best, seed, "test")][col].mean())
    print(f"{measure}: chosen on validation -> test {np.mean(scores):.1f}% (splits: {', '.join(f'{s:.1f}' for s in scores)})")
    for seed, b in zip(SPLITS, picks): print(f"     split {seed}: {b}")

et = "ExtraTrees"
crow = []
for n in CAL:
    r = {"model": n}
    for measure, col in (("one frame", 1), ("whole take", 2)):
        a = np.concatenate([PRED[(n, s, "test")][col] for s in SPLITS]); b = np.concatenate([PRED[(et, s, "test")][col] for s in SPLITS])
        better, worse = int((a & ~b).sum()), int((~a & b).sum())
        r[f"{measure}: right of {len(a)}"] = int(a.sum()); r[f"{measure}: % "] = 100 * a.mean()
        r[f"{measure}: better / worse than ExtraTrees"] = f"{better} / {worse}"
        r[f"{measure}: sign test p"] = stats.binomtest(better, better + worse).pvalue if better + worse else np.nan
    crow.append(r)
CB = pd.DataFrame(crow).set_index("model").sort_values("one frame: % ", ascending=False)
print(); print(CB.round(3).to_string())

one frame: chosen on validation -> test 94.1% (splits: 94.8, 91.4, 98.3, 96.6, 89.7)
     split 0: Neural net (PyTorch MLP)
     split 1: Neural net (PyTorch MLP), rich
     split 2: Neural net (PyTorch MLP)
     split 3: Neural net (PyTorch MLP), rich
     split 4: ExtraTrees
whole take: chosen on validation -> test 95.5% (splits: 94.8, 93.1, 98.3, 96.6, 94.8)
     split 0: Neural net (PyTorch MLP)
     split 1: Neural net (PyTorch MLP), rich
     split 2: Neural net (PyTorch MLP)
     split 3: Neural net (PyTorch MLP), rich
     split 4: ExtraTrees



                                       one frame: right of 290  one frame: %  one frame: better / worse than ExtraTrees  one frame: sign test p  whole take: right of 290  whole take: %  whole take: better / worse than ExtraTrees  whole take: sign test p
model                                                                                                                                                                                                                                                        
ExtraTrees + neural net                                    277         95.517                                    11 / 0                   0.001                       277          95.517                                      7 / 3                    0.344
Neural net (PyTorch MLP), rich                             276         95.172                                    14 / 4                   0.031                       277          95.517                                      9 / 5         

### Which letters are still missed

Test takes pooled over the five splits, whole-take scoring, for the shipped SVM, ExtraTrees and the neural network.
Only letters where at least one of the three misses a take are listed.

In [13]:
show = ["SVM-RBF (shipped), browser vote share", "ExtraTrees", "Neural net (PyTorch MLP)"]
per = {}
for n in show:
    idx = np.concatenate([PRED[(n, s, "test")][0] for s in SPLITS]); ok = np.concatenate([PRED[(n, s, "test")][2] for s in SPLITS])
    g = pd.DataFrame({"letter": y[idx], "ok": ok}).groupby("letter").ok
    per[n] = g.sum().astype(int).astype(str) + "/" + g.size().astype(str); miss = g.size() - g.sum()
    per[n + " missed"] = miss
L = pd.DataFrame(per)
L = L[L[[c for c in L if c.endswith("missed")]].sum(1) > 0]
print(L[[c for c in L if not c.endswith("missed")]].to_string())

       SVM-RBF (shipped), browser vote share ExtraTrees Neural net (PyTorch MLP)
letter                                                                          
ء                                       6/10       7/10                     8/10
ت                                       5/10       7/10                     6/10
خ                                       8/10      10/10                    10/10
ذ                                       9/10      10/10                    10/10
ر                                       6/10      10/10                    10/10
ز                                       9/10      10/10                    10/10
س                                       9/10      10/10                     9/10
ض                                       9/10      10/10                    10/10
ظ                                      10/10       9/10                    10/10
غ                                       8/10       7/10                     8/10
ق                           

## 5. What this shows

**The earlier results stand.** Every letter and word number recomputed here or by re-running the four result
notebooks matches what is recorded, to the decimal. The one exception is the shuffled-label control, which has
no seed and moves between 5.6% and 7.2%.

**A. New person (signer never seen): nothing clearly beats ExtraTrees.**

| | test accuracy | macro recall |
|---|---|---|
| ExtraTrees (reference) | 31.8% | 27.3% |
| best single new model on accuracy: ExtraTrees with cleaned labels | 33.1% | 23.7% |
| 5-model blend (geometric mean) | 33.5% | 27.2% |
| ExtraTrees, rich features + the calibration takes as extra training data | 32.1% | 28.9% |
| model chosen on the validation signer, by accuracy | 33.7% | 26.1% |
| model chosen on the validation signer, by macro recall | 33.0% | 29.0% |

* The gains are 1 to 2 points and none is statistically reliable with five signers (smallest paired p = 0.07).
  The 5-model blend is the only candidate that is ahead on accuracy without losing macro recall; it wins on 4 of 5 signers.
* The neural network, the graph network, gradient boosting and logistic regression are all **worse** than
  ExtraTrees here (by 3 to 14 points). With about 1,000 training holds and labels that are partly wrong, the
  flexible models fit the training signers and do not transfer.
* Everything sits between 30% and 34%. That ceiling is set by the data, not by the classifier: the labels are
  inferred from the videos, and five signers is very little variety. Per-signer normalisation, which is
  allowed to look at the new person's holds, adds under one point.
* The earlier scikit-learn MLP still has the highest accuracy (33.7%) and still does it by ignoring rare
  letters (macro recall 17.4%).

**B. Calibrated user: the PyTorch neural network beats ExtraTrees.**

| mean of five splits, test takes | one frame | whole take |
|---|---|---|
| SVM (shipped), browser vote share | 87.6% | 88.3% |
| ExtraTrees | 91.7% | 94.1% |
| Neural net (PyTorch MLP) | 95.2% | 95.9% |
| ExtraTrees + neural net | 95.5% | 95.5% |
| model chosen on validation | 94.1% | 95.5% |

* On single frames the neural network gets 11 test takes right that ExtraTrees gets wrong and loses 1
  (sign test p = 0.006); the blend of the two is 11 to 0. On whole takes the lead shrinks to 9 against 4,
  which is not reliable (p = 0.27): averaging over a take already removes most of ExtraTrees' errors.
* Both are well ahead of the shipped SVM: ExtraTrees fixes 21 whole-take errors of the SVM and adds 4.
* The remaining misses are concentrated in ت, پ, ء and غ.

**Limits that apply to every number in section 4.** The test takes come from the same sitting, the same two
people, the same camera and room as the training takes, and 30 letters have calibration takes while 11 have none
(آ ؤ ئ ج ش م ن ٹ ڑ ھ ہ). The 13 Nov labels have not been checked against the official signs. So 95% is
"recognising a calibrated person in the session they calibrated in". The honest test still to run is a new
recording on another day.

**How the candidates were arrived at.** Before this notebook, an exploratory round tried more settings than are
shown (several ExtraTrees and SVM settings, six PyTorch MLP configurations, self-training, LDA followed by
other classifiers, more blends) and those runs printed test scores. The candidate list here was shaped by that
round, so the test numbers of the new candidates are slightly optimistic. The choice on validation data in
sections 3 and 4 limits this but does not remove it.

**What to do with it.**
* For the live, calibrated demo, ExtraTrees or the PyTorch network would both cut the SVM's errors by about half
  on this test. Neither runs in the browser yet; the SVM is still what ships.
* For new-person accuracy, a different classifier will not help. Cleaner labels (the annotated frames) and more
  signers are what would move it.
